# LlamaIndex: Retrieval and Generation in a RAG System

To build a **Retrieval-Augmented Generation (RAG)** system, we extend retrieval by integrating a language model for response generation. Retrieval fetches relevant documents, while generation creates context-aware responses.

### This notebook covers:
- Enhancing retrieval with response generation  
- Running local inference (`llama.cpp`, Hugging Face models)  
- Using API-based inference for remote LLMs  
- Building an interactive CLI chat system  

## Retrieval and Generation in RAG

1. **Transforming Documents**: Splitting raw text into structured Nodes.  
2. **Embedding Generation**: Converting Nodes into high-dimensional vectors.  
3. **Storage & Indexing**: Storing embeddings and optimizing retrieval.  
4. **Generation**: Formatting a query-based prompt, sending it to an LLM, and refining the response.  

The quality of responses depends on well-structured retrieval, model selection, and prompt design.

## Setting Up RAG

We'll install `llama-cpp` to run open-source LLMs like **Llama 2, Mistral, and Gemma** locally.


In [137]:
%pip install llama-cpp-python
%pip install openai
%pip install llama-index-llms-llama-cpp
%pip install llama-index-postprocessor-colbert-rerank

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Next, let's rebuild our vector index. To reload the persisted index, we need to separately load the vector store and embedding model, used before persisting:

In [ ]:
%load_ext ipython_async

In [ ]:
%%asynccmd
mkdir .\vectorstore\postgres &
initdb -D .\vectorstore\postgres --encoding=UTF8 --locale=en_US.UTF-8 &
pg_ctl -D .\vectorstore\postgres -l logfile start

A subdirectory or file .\vectorstore\postgres already exists.
The files belonging to this database system will be owned by user "acisse".
This user must also own the server process.

The database cluster will be initialized with locale "en_US.UTF-8".
The default text search configuration will be set to "english".

Data page checksums are disabled.

initdb: error: directory "vectorstore/postgres" exists but is not empty
initdb: hint: If you want to create a new database system, either remove or empty the directory "vectorstore/postgres" or run initdb with an argument other than "vectorstore/postgres".
waiting for server to start...... done
server started


<Thread(Thread-27 (run_command), started daemon 16452)>

In [2]:
import os

In [3]:
db_name = "vector_db"
user = "acisse"

In [4]:
from llama_index.vector_stores.postgres import PGVectorStore

vector_store = PGVectorStore.from_params(
    database=db_name,
    host="localhost",
    password=os.getenv("PSQL_PASSWORD"),
    port=5432,
    user=user,
    table_name="physics_docs",
    embed_dim=384,
    hybrid_search=True
)

In [5]:
from llama_index.embeddings.langchain import LangchainEmbedding
from langchain.embeddings import HuggingFaceEmbeddings

# Load MiniLM embedding model (fast & efficient)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Wrap the model for LlamaIndex
embed_model = LangchainEmbedding(embedding_model)

C:\Users\acisse\AppData\Local\Temp\ipykernel_45208\3753115488.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [44]:
from pathlib import Path
from llama_index.core import StorageContext, load_index_from_storage

# Load the stored index
index_save_path = Path("assets", "llama_vector_search", "physics_documents_index")
storage_context = StorageContext.from_defaults(persist_dir=index_save_path, vector_store=vector_store)
index = load_index_from_storage(storage_context, embed_model=embed_model)

## Using a Local Python Model 

**llama.cpp** is an open-source C/C++ library designed for efficient inference of large language models (LLMs) like Llama. It enables running LLMs on systems without specialized hardware, including simple CPUs and mobile devices. Key features include:

- **Cross-Platform Support:** Compatible with x86, ARM, CUDA, Metal, Vulkan (v1.2+), and SYCL.
- **Optimized Performance:** Utilizes CPU extensions (AVX, AVX2, AVX-512 on x86-64; Neon on ARM) for enhanced efficiency.
- **Quantization:** Supports various quantization methods (2-bit to 8-bit) to reduce memory usage and improve speed, with a trade-off in precision.
- **GGUF File Format:** Employs the GGUF binary format to store tensors and metadata, facilitating rapid model loading and compatibility across different architectures.

For more details and to access the source code, visit the [llama.cpp GitHub repository](https://github.com/ggml-org/llama.cpp).

In [ ]:
from llama_index.llms.llama_cpp import LlamaCPP

model_url = "https://huggingface.co/TheBloke/Llama-2-13B-chat-GGUF/resolve/main/llama-2-13b-chat.Q4_0.gguf"
output_path = Path("assets", "llama_rag")
model_file_path = output_path / "llama-2-13b-chat.Q4_0.gguf"

llm_local = LlamaCPP(
    model_url=model_url,
    temperature=0.1,
    max_new_tokens=256,
    context_window=3900,
    generate_kwargs={},
    model_kwargs={"n_gpu_layers": 1},
    verbose=True,
)

llama_init_from_model: n_ctx_per_seq (3904) < n_ctx_train (4096) -- the full capacity of the model will not be utilized


In [ ]:
# Example usage
from utils import display_citation
prompt = "What is retrieval-augmented generation?"
display_citation(str(llm_local.complete(prompt, max_tokens=512)))

> 
> 
> Retrieval-augmented generation (RAG) is a machine learning approach that combines the strengths of both retrieval-based and generation-based models to improve the performance of open-domain question answering systems. The basic idea behind RAG is to use a retrieval model to identify relevant documents and then use a generation model to fill in the gaps in the retrieved documents.
> 
> RAG models typically consist of two main components: a retrieval model and a generation model. The retrieval model is trained to identify the most relevant documents for a given question, while the generation model is trained to generate the missing information in the retrieved documents. The retrieval model and the generation model are typically trained jointly, with the goal of improving the performance of the overall RAG system.
> 
> RAG has several potential advantages over traditional open-domain question answering systems. For example, RAG can handle complex and ambiguous questions more effectively, as it can leverage the strengths of both retrieval-based and generation-based models. Additionally, RAG can be more efficient than traditional systems, as it can use pre-trained models and avoid the need for extensive manual annotation.
> 
> There are several challenges associated with

## Using OpenAI’s API

**OpenAI API** provides cloud-based access to powerful large language models (LLMs) like GPT-4o and GPT-4 Turbo, enabling natural language processing for a variety of applications. It eliminates the need for local hardware requirements by leveraging OpenAI’s infrastructure. Key features include:

- **Scalability & Performance:**  
Runs on OpenAI’s optimized cloud servers, ensuring fast inference and handling high workloads efficiently.
- **State-of-the-Art Models:**  
Supports the latest model relases. At the time of writing this includes 4.5, 4o and o3-mini
- **Fine-Tuning & Customization:**  
Allows users to fine-tune models for domain-specific applications and customize responses via system prompts.
- **Ease of Integration:**  
Provides a simple API with support for multiple programming languages, making it easy to integrate into existing applications.

For more details and API documentation, visit the [OpenAI API reference](https://platform.openai.com/docs/).

Note, for this tutorial, we will opt for the use of OpenAI's 4o model, as it has the right balance of intelligence and price.

In [132]:
import os
import openai
from openai import OpenAI

# Load API key from environment variable
openai.api_key = os.getenv("OPENAI_API_KEY")
llm_openai = OpenAI()

def generate_response_openai(prompt):
    response = llm_openai.chat.completions.create(
        model="gpt-4o",  # Consider using the latest model
        messages=[{"role": "system", "content": "You are a helpful assistant."},
                  {"role": "user", "content": prompt}],
        max_tokens=512
    )
    return response.choices[0].message.content  # Adjusting for new response structure

# Example usage
prompt = "Explain how RAG systems work."
display_citation(str(generate_response_openai(prompt)))

> RAG systems, or Retrieval-Augmented Generation systems, are an approach in natural language processing that combines information retrieval and text generation to produce more accurate and contextually relevant responses. The core idea is to enhance the capabilities of a generative model by providing it with relevant context retrieved from external data sources. Here's a breakdown of how RAG systems typically work:
> 
> 1. **Input Query**: The process begins with an input query from a user. This query might be a question or a request for information.
> 
> 2. **Retrieval Component**: The query is first processed by a retrieval component, often based on information retrieval techniques. This component searches an external knowledge base or database to find passages or documents that are relevant to the query. The knowledge base could be as simple as a collection of documents or as complex as a structured database.
> 
> 3. **Document Scoring**: Retrieved documents or passages are scored based on their relevance to the input query. Common techniques for this include TF-IDF, BM25, or more advanced neural retrieval methods.
> 
> 4. **Selection of Relevant Context**: The top-ranked documents or passages are selected as relevant context to augment the generative process. This typically involves choosing a manageable number of documents to ensure efficiency and relevance.
> 
> 5. **Generative Component**: A generative model, such as a transformer-based language model (e.g., GPT or BERT variants), takes both the original query and the retrieved relevant contexts as input. 
> 
> 6. **Response Generation**: The generative model processes this input to generate a response. The retrieved information helps ground the generation process, providing factual accuracy and coherence that might be challenging for the model to produce on its own, particularly when the original training data lacks specific details about the query context.
> 
> 7. **Output**: The system outputs a generated response that ideally satisfies the user query by incorporating the relevant information.
> 
> RAG systems are particularly valuable for tasks requiring up-to-date information or domain-specific knowledge that might not be captured in the training data of a stand-alone language model. By integrating retrieval mechanisms, these systems benefit both from the powerful language generation capabilities of modern models and the factual grounding provided by external data sources.

Notice, that you can always opt for a different provider or model. The following models have the best performances depending on the following aspects. For more actual pricing an performance data, visit [here](https://artificialanalysis.ai/models#pricing):

- **Intelligence vs. Price:**  
  Intelligence often incurs either larger output sizes or higher network parameter counts. Both increase the price accordingly. The best performers on this ratio are **Deepseek’s R1 model** (intelligent and very cheap) and **OpenAI’s o3 model** (very intelligent and cheap).  
  <br>
  <img src="assets/llama_retrieval_augmented_generation/pics/intelligence_vs_price.png" alt="Intelligence vs. Price" width="900rem">

- **Total Response Time vs. Intelligence:**  
  In the same fashion, these aspects affect response times. Here, the winner is **Claude 3.7 Sonnet Thinking**, which has the highest intelligence with the lowest latency.  
  <br>
  <img src="assets/llama_retrieval_augmented_generation/pics/intelligence_vs_total_response_time.png" alt="Total Response Time vs. Intelligence" width="900rem">

- **Total Response Time:**  
  **Gemini Flash** is the model with the fastest response time. Its general intelligence is on par with **Claude 3.7 Sonnet (Not Thinking)**, and its pricing is in the lower end. This model is recommended for **time-critical applications**.  
  <br>
  <img src="assets/llama_retrieval_augmented_generation/pics/total_response_time.png" alt="Total Response Time" width="900rem">

## Building a Query Engine

The **Query Engine** in LlamaIndex provides a structured approach to querying indexed data. It retrieves relevant documents, applies optional filtering and ranking, and synthesizes responses using a selected LLM. This allows for **retrieval-augmented generation (RAG)**, where an LLM generates responses based on external knowledge rather than just its internal training data.

### Understanding the Components of a Query Engine

A **Query Engine** consists of three main components:  

1. **Retrieval**  
This is our vector index retrieval engine, that allows us to get the most relevant data from our files.
2. **Postprocessing**  
Filters, reranks, or augments retrieved documents for better accuracy.  
3. **Response Synthesis**  
Uses an LLM to generate a final response based on the retrieved documents.

Each of these components can be adapted to optimize retrieval, improve relevance, and refine responses.



In [171]:
from llama_index.core import Settings
from llama_index.core.query_engine import RetrieverQueryEngine

Settings.llm = None
# Assemble query engine
query_engine_raw = RetrieverQueryEngine(
    retriever=index.as_retriever(llm=None, similarity_top_k=3),
)
response = query_engine_raw.query("What is m-theory")
display_citation(response.response[:1800])


LLM is explicitly disabled. Using MockLLM.


> Context information is below.
> ---------------------
> file_path: c:\Users\acisse\OneDrive - bridgingIT-Gruppe\003_CodeWorkspace\005_roche\ai-chatdocs\notebooks\assets\extraction\markdown\eng-m-theory\converted.md
> 
> Their work shed light on the mathematical structure of M-theory and suggested possible ways of connecting M-theory to real world physics.1411
> 
> ### Origin of the term
> 
> Initially, some physicists suggested that the new theory was a fundamental theory of membranes, but Witten was skeptical of the role of membranes in the theory. In a paper from 1996, Horava and Witten wrote
> 
> As it has been proposed that the eleven-dimensional theory is a supermembrane theory but there are some reasons to doubt that interpretation, we will non-committally call it the M-theory, leaving to the future the relation of M to membranes. 1391
> 
> In the absence of an understanding of the true meaning and structure of M-theory, Witten has suggested that the M should stand for "magic", "mystery", or "membrane" according to taste, and the true meaning of the title should be decided when a more fundamental formulation of the theory is known. 11 Years later, he would state, "I thought my colleagues would understand that it really stood for membrane. Unfortunately, it got people confused. "142
> 
> ## Matrix theory
> 
> ### BFSS matrix model
> 
> In mathematics, a matrix is a rectangular array of numbers or other data. In physics, a matrix model is a particular kind of physical theory whose mathematical formulation involves the notion of a matrix in an important way. A matrix model describes the behavior of a set of matrices within the framework of quantum mechanics.[43][44]
> 
> One important example of a matrix model is the BFSS matrix model proposed by Tom Banks, Willy Fischler, Stephen Shenker, 

In [157]:
from utils import display_nodes
display_nodes(response.source_nodes, max_chars=600)

Result 3ebbaeb7-fcbe-42a6-b3eb-badbc049714a

Score: 3.4810919761657715
Start char idx: 22455; End char idx: 27087


> Their work shed light on the mathematical structure of M-theory and suggested possible ways of connecting M-theory to real world physics.1411
> 
> ### Origin of the term
> 
> Initially, some physicists suggested that the new theory was a fundamental theory of membranes, but Witten was skeptical of the role of membranes in the theory. In a paper from 1996, Horava and Witten wrote
> 
> As it has been proposed that the eleven-dimensional theory is a supermembrane theory but there are some reasons to doubt that interpretation, we will non-committally call it the M-theory, leaving to the future the relat

--------------------------------------------------
Result 54b255f3-4389-414b-8c05-ed62bc6ba715

Score: 2.1136646270751953
Start char idx: 0; End char idx: 4745


> ![](_page_0_Picture_2.jpeg)
> 
> # M-theory
> 
> M-theory is a theory in physics that unifies all consistent versions of superstring theory. Edward Witten first conjectured the existence of such a theory at a string theory conference at the University of Southern California in 1995. Witten's announcement initiated a flurry of research activity known as the second superstring revolution. Prior to Witten's announcement, string theorists had identified five versions of superstring theory. Although these theories initially appeared to be very different, work by many physicists showed that the theories

--------------------------------------------------
Result 72b97c8c-b1e7-4fd4-904e-6def173493af

Score: 0.6663401126861572
Start char idx: 3862; End char idx: 8629


> The different theories allow different types of strings, and the particles that arise at low energies exhibit different symmetries. For example, the type I theory includes both open strings (which are segments with endpoints) and closed strings (which form closed loops), while types IIA and IIB include only closed strings.12 Each of these five string theories arises as a special limiting case of M-theory, like its string theory predecessors, is an example of a quantum theory of gravity. It describes a force just like the familiar gravitational force subject to the rules of quantum mechanics.[3

--------------------------------------------------


### Node Postprocessors

Once documents are retrieved, [node postprocessing](https://docs.llamaindex.ai/en/stable/module_guides/querying/node_postprocessors/) can be applied to refine the results before they are passed to the LLM. This is particularly useful when dealing with a **large number of retrieved documents** or when relevance varies significantly.

**Similarity Filtering**:  
A common approach is **similarity filtering**, which removes results that fall below a certain relevance threshold. This helps ensure that only high-quality results are passed to the response synthesis step.

In [161]:
from llama_index.core.postprocessor import SimilarityPostprocessor

postprocessor = SimilarityPostprocessor(similarity_cutoff=0.7)

query_engine = RetrieverQueryEngine(
    retriever=index.as_retriever(llm=None, similarity_top_k=3),
    node_postprocessors=[postprocessor]
)
response = query_engine.query("What is m-theory")
display_citation(response.response[:1800])

> Empty Response

In [162]:
display_nodes(response.source_nodes, display_text=False)

In this case, only documents with a **similarity score of 0.7 or higher** are retained, reducing noise and technically improving response accuracy. However, as you can see, if selected too strictly, this might choke off all information all together.

**SentenceTransformerRerank (Open-Source):**  
Sometimes, retrieved results may not be in the optimal order. **Reranking** ensures that the most relevant documents appear at the top of the list before they are sent to the LLM. Below are different rerankers you can use.

This reranker uses pre-trained cross-encoders from the `sentence-transformers` library. Unlike embedding-based retrieval, which scores each document independently, cross-encoders score documents in the context of a query, leading to better ranking.

In [163]:
from llama_index.core.postprocessor import SentenceTransformerRerank

postprocessor = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-2-v2", 
    top_n=3 
)
query_engine = RetrieverQueryEngine(
    retriever=index.as_retriever(llm=None, similarity_top_k=3),
    node_postprocessors=[postprocessor]
)
response = query_engine.query("What is m-theory")

In [164]:
display_nodes(response.source_nodes, display_text=False)

Result 3ebbaeb7-fcbe-42a6-b3eb-badbc049714a

Score: 3.4810919761657715
Start char idx: 22455; End char idx: 27087
--------------------------------------------------
Result 54b255f3-4389-414b-8c05-ed62bc6ba715

Score: 2.1136646270751953
Start char idx: 0; End char idx: 4745
--------------------------------------------------
Result 72b97c8c-b1e7-4fd4-904e-6def173493af

Score: 0.6663401126861572
Start char idx: 3862; End char idx: 8629
--------------------------------------------------


This method does not require an API key, making it a great free alternative to Cohere.

- **Pros:** Open-source, efficient, good accuracy  
- **Cons:** Requires downloading a model (~100MB), may be slower than embedding-based retrieval  

If you need better accuracy, you can use a larger model such as `"cross-encoder/ms-marco-MultiBERT-L-12"`.

**LLMRerank (Uses OpenAI GPT-4 or Llama):**  
Instead of using a pre-trained reranking model, LLMRerank leverages an LLM to analyze and reorder retrieved documents based on their relevance.

In [165]:
from llama_index.core.postprocessor import LLMRerank
from llama_index.llms.openai import OpenAI

llm_openai=OpenAI(model="gpt-4o")

# Configure reranker
postprocessor = LLMRerank(top_n=3, llm=llm_openai)

query_engine = RetrieverQueryEngine(
    retriever=index.as_retriever(llm=None, similarity_top_k=3),
    node_postprocessors=[postprocessor]
)
response = query_engine.query("What is m-theory")

In [166]:
display_nodes(response.source_nodes, display_text=False)

Result 54b255f3-4389-414b-8c05-ed62bc6ba715

Score: 10.0
Start char idx: 0; End char idx: 4745
--------------------------------------------------
Result 3ebbaeb7-fcbe-42a6-b3eb-badbc049714a

Score: 9.0
Start char idx: 22455; End char idx: 27087
--------------------------------------------------
Result 72b97c8c-b1e7-4fd4-904e-6def173493af

Score: 8.0
Start char idx: 3862; End char idx: 8629
--------------------------------------------------


This approach **asks the LLM to rank documents** based on the query, making it highly flexible.

- **Pros:** Works well with general-purpose LLMs, highly adaptable  
- **Cons:** Requires API calls, may be expensive for large datasets  

If using OpenAI is not an option, this reranker can also work with **self-hosted Llama models** via `llama.cpp`.

**ColbertRerank (High-Precision Retrieval)**  
For scenarios where fine-grained relevance matching is needed, the **ColBERT-V2 model** can be used for re-ranking. This approach compares query tokens against document tokens, rather than scoring full document embeddings.

In [167]:
from llama_index.postprocessor.colbert_rerank import ColbertRerank

postprocessor = ColbertRerank(
    top_n=5,
    model="colbert-ir/colbertv2.0",
    tokenizer="colbert-ir/colbertv2.0",
    keep_retrieval_score=True,
)

query_engine = RetrieverQueryEngine(
    retriever=index.as_retriever(llm=None, similarity_top_k=3),
    node_postprocessors=[postprocessor]
)
response = query_engine.query("What is m-theory")

In [168]:
display_nodes(response.source_nodes, display_text=False)

Result 54b255f3-4389-414b-8c05-ed62bc6ba715

Score: 0.7366624474525452
Start char idx: 0; End char idx: 4745
--------------------------------------------------
Result 3ebbaeb7-fcbe-42a6-b3eb-badbc049714a

Score: 0.676268458366394
Start char idx: 22455; End char idx: 27087
--------------------------------------------------
Result 72b97c8c-b1e7-4fd4-904e-6def173493af

Score: 0.6160739660263062
Start char idx: 3862; End char idx: 8629
--------------------------------------------------


This method is especially useful when handling **long documents** or **complex queries** that require **token-level attention**.

- **Pros:** High accuracy, optimized for retrieval-heavy tasks  
- **Cons:** Requires a **fine-tuned reranking model**, heavier compute requirements  

In [169]:
from llama_index.core.postprocessor import LongContextReorder

postprocessor = LongContextReorder()

query_engine = RetrieverQueryEngine(
    retriever=index.as_retriever(llm=None, similarity_top_k=3),
    node_postprocessors=[postprocessor]
)
response = query_engine.query("What is m-theory")

In [170]:
display_nodes(response.source_nodes, display_text=False)

Result 3ebbaeb7-fcbe-42a6-b3eb-badbc049714a

Score: 0.5618923045332623
Start char idx: 22455; End char idx: 27087
--------------------------------------------------
Result 54b255f3-4389-414b-8c05-ed62bc6ba715

Score: 0.48848414421081543
Start char idx: 0; End char idx: 4745
--------------------------------------------------
Result 72b97c8c-b1e7-4fd4-904e-6def173493af

Score: 0.4936602711677551
Start char idx: 3862; End char idx: 8629
--------------------------------------------------



This is especially useful when **retrieving large chunks of text** but needing the most crucial information **at the top**.

## Response Synthesis

After retrieval and postprocessing, the selected documents are passed to an LLM, which synthesizes a final response. LlamaIndex offers multiple **response modes**, each designed for different tasks.

First let's consider the response without synthesis:

In [173]:
response = query_engine_raw.query("What is m-theory")
display_citation(response.response[:1000])

> Context information is below.
> ---------------------
> file_path: c:\Users\acisse\OneDrive - bridgingIT-Gruppe\003_CodeWorkspace\005_roche\ai-chatdocs\notebooks\assets\extraction\markdown\eng-m-theory\converted.md
> 
> Their work shed light on the mathematical structure of M-theory and suggested possible ways of connecting M-theory to real world physics.1411
> 
> ### Origin of the term
> 
> Initially, some physicists suggested that the new theory was a fundamental theory of membranes, but Witten was skeptical of the role of membranes in the theory. In a paper from 1996, Horava and Witten wrote
> 
> As it has been proposed that the eleven-dimensional theory is a supermembrane theory but there are some reasons to doubt that interpretation, we will non-committally call it the M-theory, leaving to the future the relation of M to membranes. 1391
> 
> In the absence of an understanding of the true meaning and structure of M-theory, Witten has suggested that the M should stand for "magic", "mystery", or "me

A basic response synthesizer can be initialized as follows. The response modes affect the generation of the prompt in the following way:
- `"compact"`  
  This mode merges as much relevant content as possible into a single LLM prompt, reducing the number of API calls. It is useful when efficiency is a priority but can lose some detail if the document set is too large.  

- `"refine"`  
  The LLM first generates an initial response using the most relevant document, then iteratively refines it as more context is introduced. This approach produces highly accurate answers but is slower due to multiple LLM calls.  

- `"tree_summarize"`  
  Information is summarized in a hierarchical fashion, where smaller summaries are merged into a final response. This method is effective for long documents but can risk excessive compression if too many layers are applied.  

- `"simple_summarize"`  
  All retrieved text is concatenated and summarized in a single LLM call. It is the fastest approach but fails if the total text size exceeds the LLM's context window.  

- `"generation"`  
  The response is generated entirely by the LLM without using retrieved documents. This mode works well for general knowledge queries but lacks grounding in external sources.  

- `"no_text"`  
  The engine returns the raw retrieved documents instead of synthesizing a final response. This is useful for debugging or when external processing is needed.  

- `"context_only"`  
  The retrieved text is concatenated and returned as-is, without LLM processing. This ensures full transparency but requires manual interpretation.  

- `"accumulate"`  
  A response is generated for each text chunk individually, and the results are concatenated. This mode preserves more details but can lead to repetitive responses when content overlaps.  

- `"compact_accumulate"`  
  Text chunks are merged into larger blocks before the LLM processes them, balancing efficiency and detail. While it reduces redundancy, some fine-grained information may be lost during consolidation.  

Each mode offers trade-offs between cost, response depth, and efficiency, making it important to choose based on the use case.

In [191]:
response_mode = "compact"
response_mode = "refine"
response_mode = "tree_summarize"

In [ ]:
from llama_index.core import get_response_synthesizer

response_synthesizer = get_response_synthesizer(
    response_mode=response_mode, 
    llm=llm_openai
)

query_engine = RetrieverQueryEngine(
    retriever=index.as_retriever(llm=None, similarity_top_k=3),
    response_synthesizer=response_synthesizer
)
response = query_engine.query("What is m-theory")
display_citation(response.response[:1000])

> M-theory is a theoretical framework in physics that aims to unify all consistent versions of superstring theory. It was first conjectured by Edward Witten in 1995 and has since been a central focus of research in theoretical physics. M-theory suggests that the different string theories are related through mathematical transformations known as dualities, specifically S-duality and T-duality. Although a complete formulation of M-theory is not yet known, it is expected to describe two- and five-dimensional objects called branes and to be approximated by eleven-dimensional supergravity at low energies. M-theory is considered a potential framework for developing a unified theory of all fundamental forces of nature, and it involves extra dimensions that are often compactified to model the four-dimensional world we observe.

## Customizing Response Synthesis with Templates

When generating responses, LlamaIndex provides several ways to customize how the retrieved documents are processed and presented to the LLM. Three key components influence this process:  

1. **`text_qa_template`**  
Specifies how retrieved documents are formatted before being sent to the LLM. By structuring the prompt carefully, this template helps improve the accuracy and clarity of responses.  
2. **`refine_template`**  
Used in **refine mode**, this template enables an iterative refinement process where the LLM starts with an initial answer and improves it as more documents are processed.  

Each of these elements allows for fine-grained control over response synthesis. Whether structuring an initial query, refining an evolving answer, or optimizing for efficiency, **custom templates help tailor the interaction between retrieval and generation**. 

For the text QA template, the default prompt goes something like:

```bash
    Context information is below.
    ---------------------
    {context_str}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    Query: {query_str}
    Answer:
```
Let's look at the prompt object itself. While complex in it's make up, we do not require much to replicate it

In [187]:
response_synthesizer.get_prompts()["text_qa_template"].dict()

{'metadata': {'prompt_type': <PromptType.QUESTION_ANSWER: 'text_qa'>},
 'template_vars': ['context_str', 'query_str'],
 'kwargs': {},
 'output_parser': None,
 'template_var_mappings': {},
 'function_mappings': {},
 'default_template': {'metadata': {'prompt_type': <PromptType.QUESTION_ANSWER: 'text_qa'>},
  'template_vars': ['context_str', 'query_str'],
  'kwargs': {},
  'output_parser': None,
  'template_var_mappings': None,
  'function_mappings': None,
  'template': 'Context information is below.\n---------------------\n{context_str}\n---------------------\nGiven the context information and not prior knowledge, answer the query.\nQuery: {query_str}\nAnswer: '},
 'conditionals': [(<function llama_index.core.prompts.utils.is_chat_model(llm: llama_index.core.base.llms.base.BaseLLM) -> bool>,
   {'metadata': {'prompt_type': <PromptType.CUSTOM: 'custom'>},
    'template_vars': ['context_str', 'query_str'],
    'kwargs': {},
    'output_parser': None,
    'template_var_mappings': None,
    

Let's try it out. We'll build the following two prompt objects:

In [194]:
from llama_index.core import PromptTemplate


# Define the QA template
text_qa_template = PromptTemplate(
    "Given the following context, answer the question in a dunny and"
    " light harted way. Use the word llama-index in every sentense.\n\n"
    "Context:\n{context_str}\n\n"
    "Question: {query_str}\n\n"
    "Answer:"
)

# Define the refinement template
refine_template = PromptTemplate(
    "We have an existing answer: {existing_answer}\n\n"
    "Given the following additional context, refine the answer.\n\n"
    "Context:\n{context_str}\n\n"
    "Updated Answer:"
)

By now injecting the prompts into the synthesize, we can modify the genration:

In [198]:
from llama_index.core import get_response_synthesizer

response_synthesizer = get_response_synthesizer(
    response_mode="compact", 
    llm=llm_openai,
    text_qa_template=text_qa_template,
    refine_template=refine_template
)

query_engine = RetrieverQueryEngine(
    retriever=index.as_retriever(llm=None, similarity_top_k=3),
    response_synthesizer=response_synthesizer
)
response = query_engine.query("What is m-theory")
display_citation(response.response[:1000])

> Ah, M-theory, the llama-index of the physics world! Imagine a universe where all the superstring theories are having a grand party, and M-theory is the llama-index that unites them all under one cosmic dance floor. Llama-index Witten, the party planner, first introduced this concept in 1995, and since then, physicists have been trying to figure out if the "M" in M-theory stands for "magic," "mystery," or "membrane"—or maybe it's just the llama-index of all three!
> 
> In the world of M-theory, dimensions are like layers of a llama-index cake, with eleven dimensions to explore. It's a place where strings and branes do the llama-index tango, and gravity and quantum mechanics try to get along like old friends at a llama-index reunion. So, in short, M-theory is the llama-index of theoretical physics, promising to unlock the secrets of the universe, one llama-index at a time!

## Quickly Creating the Query Engine formt the Index

For quick and efficient querying, LlamaIndex provides the `.as_query_engine()` method. This high-level API allows users to retrieve information from an indexed dataset without needing to manually configure retrieval, postprocessing, or response synthesis. It is ideal for cases where default settings are sufficient or when rapid prototyping is required.  

This method creates a query engine directly from an index, automatically handling retrieval, postprocessing and response synthesis.
By default, `.as_query_engine()` uses OpenAI’s GPT models (if an API key is set).


In [202]:
Settings.llm = llm_openai
query_engine = index.as_query_engine()
response = query_engine.query("What is quantum mechanics?")
display_citation(response.response)

> Quantum mechanics is a fundamental theory in physics that describes the physical properties of nature at the scale of atoms and subatomic particles. It provides a mathematical framework for understanding the behavior and interactions of particles, incorporating principles such as wave-particle duality and the uncertainty principle.

You can customize the RAG using the following parameters:

In [206]:
Settings.llm = llm_openai
query_engine = index.as_query_engine(
    response_mode="refine",
    similarity_top_k=5,
    node_postprocessors=[postprocessor],
    response_synthesizer=response_synthesizer
)
response = query_engine.query("What is quantum mechanics?")
display_citation(response.response)


> Ah, quantum mechanics, the llama-index of the physics world! Imagine a universe where particles dance the llama-index tango, sometimes behaving like waves, sometimes like particles, and always keeping you guessing. In this llama-index realm, uncertainty is the name of the game, and particles can be in multiple places at once, like a llama-index magician's trick. It's a world where Schrödinger's cat might just be a llama-index enthusiast, both alive and dead, until you take a peek. So, if you're ready to embrace the llama-index chaos and the mind-bending wonders of the quantum world, welcome to the quirky, unpredictable, and utterly fascinating universe of quantum mechanics!


## Building a CLI Chat with RAG
To make **RAG interactive**, we’ll build a **command-line chat interface** that:

In [66]:
use_open_ai = True          # Set false to use the local Llama2
show_system_prompts = True  # Set false to ommit system prompts when chatting 
similarity_top_k = 10
similarity_cutoff = 0.6

In [ ]:
from dotenv import load_dotenv
from llama_index.core import get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor

load_dotenv()

openai.api_key = os.getenv('OPENAI_API_KEY')

def query_rag(prompt, use_openai=False):
    """Queries the RAG system with a given prompt."""

    # Configure retriever
    retriever = VectorIndexRetriever(index=index, similarity_top_k=similarity_top_k)

    # Configure response synthesizer
    response_synthesizer = get_response_synthesizer()

    # Configure postprocessors
    node_postprocessors = [SimilarityPostprocessor(similarity_cutoff=similarity_cutoff)]

    # Assemble query engine
    query_engine = RetrieverQueryEngine(
        retriever=retriever,
        response_synthesizer=response_synthesizer,
        node_postprocessors=node_postprocessors,
    )

    # Retrieve relevant documents
    retrieved_docs = query_engine.query(prompt)
    context = retrieved_docs.response \
        if retrieved_docs and retrieved_docs.response and not retrieved_docs.response == "Empty Response" \
        else "No relevant documents found."

    # Format final prompt
    final_prompt = (f"You are a question answering chatbot and you are conversing with the user."
                    f" Based on the following context:\n{context}\n\nAnswer the prompt comming in from the user: {prompt}")
    if show_system_prompts:
        print("-"*50)
        print("System prompt:")
        display_citation(final_prompt)
        print("-"*50)

    if use_openai:
        return generate_response_openai(final_prompt)
    else:
        return str(llm_local.complete(final_prompt, max_tokens=512))

def interactive_chat(use_openai=False):
    """Interactive chat mode in CLI."""
    print("\n Welcome to our little CLI RAG Chat experiment! Type 'exit' to quit.\n")

    while True:
        user_input = input("\nUser: ").strip()
        print("\nUser: ")
        display_citation(user_input)
        if user_input.lower() == "exit":
            print("Goodbye!")
            break

        response = query_rag(user_input, use_openai)
        print("\nAssistant:")
        display_citation(response)


interactive_chat(True)



 Welcome to our little CLI RAG Chat experiment! Type 'exit' to quit.


User: 


> do you have knwoledege on string theorx

--------------------------------------------------
System prompt:


> You are a question answering chatbot and you are conversing with the user. Based on the following context:
> No relevant documents found.
> 
> Answer the prompt comming in from the user: do you have knwoledege on string theorx

--------------------------------------------------

Assistant:


> It seems like you are asking about string theory. Yes, I can provide information on that topic! String theory is a theoretical framework in physics that attempts to reconcile quantum mechanics and general relativity. It suggests that the fundamental constituents of the universe are not point particles, but rather tiny, one-dimensional strings. These strings can vibrate at different frequencies, and the vibrations correspond to various elementary particles. The theory also predicts extra dimensions beyond the familiar three dimensions of space and one of time. String theory remains one of the candidates for a theory of everything, a single framework that encompasses all forces and particles of nature, although it is still highly theoretical and not yet fully proven or accepted universally. If you have more specific questions about string theory, feel free to ask!


User: 


> exit

Goodbye!
